# Humedad: horizonte de 120 segundos

Experimento separado de S10. Se predice la media del minuto t+2 usando las medias del minuto t. Los pares requieren que la segunda fila posterior del mismo dispositivo y tramo tenga una separación exacta de 120 s. No se rellenan huecos. Cortes cronológicos 60/20/20, persistencia, regresión lineal y Random Forest; selección en validación, test independiente y recarga del modelo.

El horizonte se mide entre los inicios de los minutos, no desde la publicación. Si t queda disponible en t+70 s, quedan aproximadamente 50 s hasta que empiece el minuto objetivo; se medirá el margen real. Una predicción emitida después de ese inicio se marca tardía y no cuenta como anticipación.

La evaluación usa Gold histórico completo. Los minutos vivos pueden estar incompletos por el plazo de recepción: estas métricas no acreditan todavía rendimiento con ese cambio de distribución.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, math
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
spark=(SparkSession.builder.appName("AireIQ-S10-Pronostico").master("local[2]")
       .config("spark.sql.session.timeZone","UTC").config("spark.sql.shuffle.partitions","2").getOrCreate())
root=Path('/evidence') if Path('/evidence').exists() else Path('/opt/aireiq/outputs')
sources=sorted(p for p in (root/'s03').iterdir() if (p/'manifest.json').exists() and (p/'gold/_SUCCESS').exists())
assert sources
source=sources[-1]
source_manifest=json.loads((source/'manifest.json').read_text())
out=root/'s10_h120'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True,exist_ok=False)
data=spark.read.parquet(str(source/'gold'))
w=Window.partitionBy('dispositivo','tramo').orderBy('minuto')
with_next=(data.withColumn('target_time',F.lead('minuto',2).over(w))
    .withColumn('label',F.lead('humedad_pct',2).over(w)))
pairs=with_next.filter(F.col('target_time').cast('long')-F.col('minuto').cast('long')==120).cache()
assert pairs.count()>100
times=[r.minuto for r in pairs.select('minuto').distinct().orderBy('minuto').collect()]
cut1,cut2=times[int(len(times)*.6)],times[int(len(times)*.8)]
train=pairs.filter(F.col('target_time')<F.lit(cut1)).cache()
validation=pairs.filter((F.col('minuto')>=F.lit(cut1))&(F.col('target_time')<F.lit(cut2))).cache()
test=pairs.filter(F.col('minuto')>=F.lit(cut2)).cache()
counts={k:v.count() for k,v in [('train',train),('validation',validation),('test',test)]}
assert min(counts.values())>0
assert train.agg(F.max('target_time')).first()[0]<validation.agg(F.min('minuto')).first()[0]
assert validation.agg(F.max('target_time')).first()[0]<test.agg(F.min('minuto')).first()[0]
print('Minutos Gold:',data.count(),'Pares consecutivos:',pairs.count(),'Bloques:',counts)
print('Pares excluidos en cortes:',pairs.count()-sum(counts.values()))


In [ ]:
features=['humedad_pct','temperatura_c','presion_hpa','hora_sin','hora_cos']
time_sql="SELECT *, sin(2*PI()*(hour(from_utc_timestamp(minuto,'America/Lima'))+minute(minuto)/60.0)/24) AS hora_sin, cos(2*PI()*(hour(from_utc_timestamp(minuto,'America/Lima'))+minute(minuto)/60.0)/24) AS hora_cos FROM __THIS__"
def model(name):
    if name=='persistencia':
        return Pipeline(stages=[SQLTransformer(statement='SELECT *, humedad_pct AS prediction FROM __THIS__')])
    estimator={
        'lineal':LinearRegression(labelCol='label',regParam=.1,maxIter=100),
        'random_forest':RandomForestRegressor(labelCol='label',numTrees=50,maxDepth=6,seed=42)
    }[name]
    return Pipeline(stages=[SQLTransformer(statement=time_sql),VectorAssembler(inputCols=features,outputCol='features',handleInvalid='error'),estimator])
def metrics(pred):
    result={m:RegressionEvaluator(labelCol='label',predictionCol='prediction',metricName=m).evaluate(pred) for m in ['rmse','mae','r2']}
    assert all(math.isfinite(v) for v in result.values())
    return result
results=[]
for name in ['persistencia','lineal','random_forest']:
    fitted=model(name).fit(train)
    pred=fitted.transform(validation).cache()
    score=metrics(pred); results.append({'modelo':name,**score})
    print(name,score,flush=True);pred.unpersist()
# Desempate a favor de persistencia, evitando complejidad sin mejora.
winner=min(results,key=lambda r:(r['rmse'],r['modelo']!='persistencia'))['modelo']
spark.createDataFrame(results).select('modelo','rmse','mae','r2').orderBy('rmse').show()
print('Elegido por validación:',winner)


In [ ]:
development=train.unionByName(validation).cache()
final_model=model(winner).fit(development)
pred=final_model.transform(test).cache()
test_score=metrics(pred)
baseline_score=metrics(test.withColumn('prediction',F.col('humedad_pct')))
final_model.write().save(str(out/'modelo'))
loaded=PipelineModel.load(str(out/'modelo'))
again=loaded.transform(test).select('dispositivo','minuto',F.col('prediction').alias('restored'))
diff=pred.select('dispositivo','minuto','prediction').join(again,['dispositivo','minuto']).agg(F.max(F.abs(F.col('prediction')-F.col('restored')))).first()[0]
assert diff is not None and diff<=1e-9
test.select('dispositivo','minuto','target_time','humedad_pct','temperatura_c','presion_hpa','label').write.parquet(str(out/'test_pairs'))
pred.select('dispositivo','minuto','target_time','humedad_pct','label','prediction').write.parquet(str(out/'predicciones_test'))
relative=100*(baseline_score['rmse']-test_score['rmse'])/baseline_score['rmse'] if baseline_score['rmse'] else None
report=dict(status='trained_and_verified',s03_run=source.name,source_sha256=source_manifest['source_sha256'],
    run_id=out.name,objetivo='humedad media del minuto que inicia 120 segundos despues del minuto de entrada',horizonte_segundos=120,
    unidad_entrada='promedio por minuto',spark_version=spark.version,timezone='UTC; hora ciclica America/Lima',
    total_minutos=data.count(),pares_consecutivos=pairs.count(),conteos=counts,
    pares_cruzando_cortes_excluidos=pairs.count()-sum(counts.values()),features=features,
    validacion=results,modelo_elegido=winner,test=test_score,baseline_test=baseline_score,
    mejora_rmse_test_porcentaje=relative,diferencia_recarga=diff,
    test_start=str(test.agg(F.min('minuto')).first()[0]),test_end=str(test.agg(F.max('target_time')).first()[0]))
(out/'manifest.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(report,indent=2,ensure_ascii=False))
print('Modelo y evidencia:',out)
for frame in [pairs,train,validation,test,development,pred]:frame.unpersist()
